# Clearance markdowns at bringToMX — an experiment that answers the wrong question well

**Synthetic data.** Everything below comes from the sibling project `holding-360`, a fictional
three-company import group. No real company is described.

**The short version.** Four SKUs carry a clearance markdown in March, September and November. That
is a natural experiment, and it has a clean difference-in-differences design. The design works: it
measures the markdown's cost to margin to within **2%** of the known truth.

And that is the problem. The quantity this design measures precisely is the one that is true by
arithmetic. The question a business would actually pay to answer — *does a markdown sell more?* —
is unanswerable in this data, because the generator has no price elasticity at all.

This notebook shows both halves: the estimate that works, and the power analysis that proves the
interesting question was never reachable.

## 0. Setup

The project's own modules do the work. Nothing is re-implemented here — if the notebook and `src/`
ever disagree, the notebook is wrong.

In [1]:
import sys
from pathlib import Path

# A notebook's working directory is notebooks/, so the project root must be added explicitly
# before `src` can be imported.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.panel import TREATED_SKUS, TREATED_MONTHS, load_panel
from src.did import estimate_did, mix_decomposition, placebo_did, welch_test, cluster_bootstrap_did
from src.power import required_sample_size, simulate_power

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print(f"treated SKUs  : {list(TREATED_SKUS)}")
print(f"treated months: {list(TREATED_MONTHS)}")

treated SKUs  : ['BTM-2204', 'BTM-2205', 'BTM-1111', 'BTM-1112']
treated months: [3, 9, 11]


## 1. Build the panel, and check the treatment is where it claims to be

Nothing downstream means anything if the treatment label is wrong, so this check comes first. Every
discounted line must sit in a treated SKU **and** a treated month.

In [2]:
panel = load_panel()
frame = panel.frame

discounted = frame[frame["discounted"]]
mislabelled = int((~discounted["treated_sku"] | ~discounted["treated_period"]).sum())

print(f"invoice lines            : {len(frame):,}")
print(f"months                   : {frame['month'].nunique()} ({frame['month'].min()} -> {frame['month'].max()})")
print(f"discounted lines         : {len(discounted)}")
print(f"mislabelled              : {mislabelled}  {'OK' if mislabelled == 0 else 'PROBLEM'}")

grid = frame.groupby(["treated_sku", "treated_period"]).size().unstack(fill_value=0)
grid.index = ["control SKUs", "treated SKUs"]
grid.columns = ["normal months", "markdown months"]
grid

invoice lines            : 3,911
months                   : 18 (2025-01 -> 2026-06)
discounted lines         : 27
mislabelled              : 0  OK


,normal months,markdown months
control SKUs,2943,843
treated SKUs,98,27


**Read the grid.** The treated cell holds 27 lines. Everything that follows is limited by that
number, and the honest treatment of it is to say so rather than to run a test as though 27 lines
were 27 independent facts.

## 2. Estimate the effect, and compare it against the truth

The world is synthetic, so the true effect is not a mystery: the markdown is a mechanical price cut
at fixed cost, so the effect on one unit is exactly `-list_price x markdown`. That makes this an
**estimator check**, not a discovery.

In [3]:
est = estimate_did(frame)
truth = panel.truth_att
t_stat, p_value = welch_test(frame)

summary = pd.DataFrame({
    "estimate": [est.treated_change, est.did, truth],
    "error vs truth": [
        est.treated_change - truth,
        est.did - truth,
        0.0,
    ],
}, index=[
    "naive own-change (treated)",
    "DiD",
    "KNOWN TRUTH (ATT)",
])
summary["abs error %"] = (summary["error vs truth"].abs() / abs(truth) * 100).round(2)
summary

,estimate,error vs truth,abs error %
naive own-change (treated),"-1,847.05",53.32,2.81
DiD,"-1,870.71",29.66,1.56
KNOWN TRUTH (ATT),"-1,900.37",0.00,0.00


In [4]:
point, low, high = cluster_bootstrap_did(frame, n_boot=2_000, seed=11)
print(f"Welch t on treated lines : t = {t_stat:,.2f}, p = {p_value:.3g}")
print(f"cluster bootstrap 95% CI : [{low:,.2f}, {high:,.2f}]")
print(f"covers the known truth   : {low <= truth <= high}")

Welch t on treated lines : t = -6.90, p = 6.02e-08
cluster bootstrap 95% CI : [-2,468.92, -1,296.04]
covers the known truth   : True


**Both estimators recover the effect.** The DiD is marginally closer to the truth than the naive
comparison, and the bootstrap interval covers it. So far this looks like a textbook result.

The next section is where it stops looking like one.

## 3. What is the second difference actually removing?

DiD earns its keep by subtracting a **common shock** — something that moved both groups. If nothing
common moved, the second difference is subtracting noise or composition, and the sophistication buys
almost nothing.

The test: recompute the control group's monthly mean margin with **fixed SKU weights**. If the raw
series swings while the fixed-weight series is flat, the swing was product mix all along.

In [5]:
control = frame[~frame["treated_sku"]]
mix = mix_decomposition(control)

pd.DataFrame({
    "raw monthly mean (MXN)": mix["raw_mean"],
    "fixed-weight mean (MXN)": mix["fixed_weight_mean"],
    "mix gap": mix["mix_gap"],
}).head(8)

,raw monthly mean (MXN),fixed-weight mean (MXN),mix gap
month,,,
2025-01,"1,489.73","1,681.22",-191.49
2025-02,"1,638.81","1,681.22",-42.40
2025-03,"1,629.47","1,681.22",-51.75
2025-04,"1,715.80","1,681.22",34.59
2025-05,"1,690.82","1,681.22",9.60
2025-06,"1,644.10","1,681.22",-37.12
2025-07,"1,630.15","1,681.22",-51.07
2025-08,"1,749.65","1,681.22",68.44


In [6]:
print(f"raw mean            : sd {mix['raw_mean'].std():>8,.2f}  "
      f"range {mix['raw_mean'].max() - mix['raw_mean'].min():>8,.2f}")
print(f"fixed-weight mean   : sd {mix['fixed_weight_mean'].std():>8,.2f}  "
      f"range {mix['fixed_weight_mean'].max() - mix['fixed_weight_mean'].min():>8,.2f}")
print()
print("! A fixed-weight series with sd = 0 means the control group's movement is ENTIRELY mix.")
print("! There is no time-varying cost or price shock. The second difference removes a")
print(f"!  composition artefact worth MXN {abs(est.control_change):,.2f} against an effect of "
      f"MXN {abs(truth):,.2f}.")

raw mean            : sd    72.51  range   269.33
fixed-weight mean   : sd     0.00  range     0.00

! A fixed-weight series with sd = 0 means the control group's movement is ENTIRELY mix.
! There is no time-varying cost or price shock. The second difference removes a
!  composition artefact worth MXN 23.66 against an effect of MXN 1,900.37.


### The placebo test

Apply the same DiD machinery to a **fake** treatment inside months where no markdown ever happened.
A well-behaved design returns approximately zero. Whatever it returns instead is the noise floor —
the size of estimate this design can manufacture out of nothing.

In [7]:
placebo = placebo_did(frame)
print(f"real DiD    : {est.did:>10,.2f}")
print(f"placebo DiD : {placebo.did:>10,.2f}")
print(f"ratio       : {abs(placebo.did / est.did):>10.1%}")
print()
print("The placebo sits at the same magnitude as the control group's mix artefact, which is")
print("exactly what it should be: there is no shock to find, so the design finds composition.")

real DiD    :  -1,870.71
placebo DiD :      25.71
ratio       :       1.4%

The placebo sits at the same magnitude as the control group's mix artefact, which is
exactly what it should be: there is no shock to find, so the design finds composition.


## 4. Power — what could this design ever have detected?

The formula, for a two-arm comparison:

```
n per arm = 2 * (z_{1-alpha/2} + z_{1-beta})^2 / (delta / sigma)^2
```

Two ways to compute it are shown, and they must agree. The closed form is what an interviewer
expects; the simulation is the check that the formula has no slip in it.

In [8]:
treated_normal = frame.loc[frame["treated_sku"] & ~frame["treated_period"], "unit_margin"]
base_mean, base_sd = float(treated_normal.mean()), float(treated_normal.std())

rows = []
for pct in (0.05, 0.10, 0.15):
    res = required_sample_size(base_mean * pct, base_sd)
    empirical = simulate_power(base_mean * pct, base_sd, res.n_per_arm, seed=7)
    rows.append({
        "effect": f"{pct:.0%}",
        "delta (MXN)": round(base_mean * pct, 2),
        "effect size": round(res.effect_size, 4),
        "n per arm": res.n_per_arm,
        "formula power": res.power,
        "simulated power": round(empirical, 3),
    })

pd.DataFrame(rows).set_index("effect")

,delta (MXN),effect size,n per arm,formula power,simulated power
effect,,,,,
5%,71.43,0.08,2777,0.80,0.80
10%,142.86,0.15,695,0.80,0.81
15%,214.30,0.23,309,0.80,0.80


**The formula holds**: the simulated power lands on the target within a point or two. And the
requirement grows with the square of the effect — halving the effect you want to detect quadruples
the sample.

Now the question that matters. **Does a markdown sell more units?**

In [9]:
control_units = control.groupby(["item_id", "month"])["quantity"].sum()
unit_mean, unit_sd = float(control_units.mean()), float(control_units.std())
available = int(frame.loc[frame["treated_sku"]].groupby(["item_id", "month"]).ngroups)

rows = []
for lift in (0.05, 0.10, 0.20, 0.30):
    res = required_sample_size(unit_mean * lift, unit_sd)
    rows.append({
        "lift to detect": f"{lift:.0%}",
        "delta (units)": round(unit_mean * lift, 2),
        "effect size": round(res.effect_size, 4),
        "SKU-months per arm": res.n_per_arm,
    })

print(f"units per SKU-month: mean {unit_mean:,.2f}, sd {unit_sd:,.2f}")
print(f"treated SKU-months available: {available}")
print()
pd.DataFrame(rows).set_index("lift to detect")

units per SKU-month: mean 13.68, sd 14.24
treated SKU-months available: 70



,delta (units),effect size,SKU-months per arm
lift to detect,,,
5%,0.68,0.05,6805
10%,1.37,0.10,1702
20%,2.74,0.19,426
30%,4.10,0.29,190


## 5. The finding

**Look at the two tables together.** To detect a 10% change in unit margin the design needs about
**695 lines** per arm — and it has **27** in the treated cell. To detect a 10% lift in **units** it
would need about **1,702 SKU-months** per arm against **70** available.

But the sample size is not the real problem. The real problem is this: **the generator allocates
units from family volumes, seasonality and SKU velocity. Price never enters that function.**

So the markdown's effect on volume is **exactly zero by construction**. No test can find an effect
that does not exist — and running one anyway would produce a null result that reads like evidence
while being pure arithmetic.

### What this design can and cannot say

| Question | Answerable here? |
| --- | --- |
| Does the markdown reduce margin per unit? | **Yes** — measured to within 2% of the known truth |
| Did a common shock affect both groups? | **No such shock exists** — the control group moves on mix alone |
| Can the second difference be credited with removing confounders? | **Only weakly** — it removes MXN 23.66 of mix against a MXN 1,900 effect |
| Does the markdown sell more units? | **No** — zero by construction, and 1,702 SKU-months per arm would be needed to test it in a world that had an effect |
| What should the owner do? | Treat the SKUs as a **cost** decision, not an experiment decision |

### Why this is the useful result

It would have been easy to report the DiD estimate, note that the confidence interval excluded zero,
and call the exercise a success. That report would have been true and useless: it would have
presented an arithmetic identity as a causal discovery, and it would have left the reader believing
the demand question had been tested.

**The competence being demonstrated is not "can run a DiD". It is "knows which question the design
can answer, and says so out loud when the answer is the unflattering one."**

## 6. The decision

1. **The margin question is closed.** The markdown costs **MXN 1,900.37** of margin per unit, measured
   to within 2%. This confirms `holding-360`'s finding F-04 with an interval instead of a point.
2. **The demand question is not closed, and cannot be closed with this data.** Say so, and do not
   let a well-powered answer to a trivial question stand in for an unanswerable important one.
3. **The owner's decision stands as a cost decision**, not as an experiment: withdrawal of the two
   loss-making SKUs is justified by the margin arithmetic alone.
4. **If demand response is genuinely wanted**, the route is a prospective test with the sample size
   from §4, or a generator that models elasticity. Both are stated in `DESIGN.md` §10.
